# Notebook para la limpieza y auditoría de lecturas meteorológicas

## Criterios de limpieza

- Dos lecturas son duplicadas si todos sus campos, excepto `id`, son iguales. Se conserva la primera.
- Las medidas fuera de un rango físico razonable se consideran ausentes.
- Solo se imputan ausencias aisladas entre dos lecturas válidas de la misma estación separadas exactamente cinco minutos.
- Las filas que siguen incompletas se eliminan del conjunto limpio.
- Los huecos, ceros sospechosos y saltos bruscos se marcan, pero no se borran automáticamente: pueden corresponder a un evento meteorológico o de red real.

In [1]:
import json
import numpy as np
import pandas as pd

## Rutas, columnas y umbrales

El notebook se ejecuta desde `notebooks/`, por eso las rutas comienzan con `../data/`. El JSON original nunca se modifica: la salida se guarda en `data/processed/0_cleaning/` para que el resto del flujo use una versión limpia y trazable.

En este bloque se definen las columnas obligatorias, el intervalo esperado de cinco minutos y los límites físicos básicos de cada medición. Si falta una columna requerida, el proceso se detiene para no producir un dataset incompleto. Los límites separan mediciones imposibles de saltos que pueden ser un evento real.

In [2]:
INPUT_PATH = '../data/raw/readings/aruba_readings_20260924.json'
CSV_OUTPUT_PATH = '../data/processed/0_cleaning/aruba_readings_cleaned.csv'
JSON_OUTPUT_PATH = '../data/processed/0_cleaning/aruba_readings_cleaned.json'
REPORT_OUTPUT_PATH = '../data/processed/0_cleaning/data_cleaning_report.json'

ID_COLUMNS = ['id', 'station_id', 'timestamp']
NUMERIC_COLUMNS = [
    'temperature_c', 'humidity_pct', 'wind_speed_kmh', 'wind_direction_deg',
    'pressure_hpa', 'precipitation_mm', 'visibility_km', 'uv_index',
]
REQUIRED_COLUMNS = set(ID_COLUMNS + NUMERIC_COLUMNS)
EXPECTED_INTERVAL = pd.Timedelta('5min')

# Conservative limits: values outside them are treated as invalid measurements.
VALID_RANGES = {
    'temperature_c': (10, 45),
    'humidity_pct': (0, 100),
    'wind_speed_kmh': (0, 180),
    'wind_direction_deg': (0, 360),
    'pressure_hpa': (850, 1100),
    'precipitation_mm': (0, 150),
    'visibility_km': (0, 100),
    'uv_index': (0, 15),
}

# A jump above these values in five minutes is kept and flagged for review.
MAX_CHANGE_PER_INTERVAL = {
    'temperature_c': 5, 'humidity_pct': 30, 'wind_speed_kmh': 60,
    'wind_direction_deg': 180, 'pressure_hpa': 15, 'precipitation_mm': 30,
    'visibility_km': 10, 'uv_index': 5,
}

## Carga y normalización de tipos

Leemos el JSON original sin modificarlo. Las fechas inválidas y los valores no numéricos se convierten en ausentes para tratarlos de forma explícita en los pasos siguientes.

In [3]:
raw_df = pd.read_json(INPUT_PATH)
missing_columns = REQUIRED_COLUMNS.difference(raw_df.columns)
if missing_columns:
    raise KeyError(f'Faltan columnas requeridas: {sorted(missing_columns)}')

df = raw_df.copy()
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
df[NUMERIC_COLUMNS] = df[NUMERIC_COLUMNS].apply(pd.to_numeric, errors='coerce')

print(f'Filas originales: {len(df):,}')
print(f'Fechas inválidas: {df["timestamp"].isna().sum():,}')
df.head()

Filas originales: 1,351,008
Fechas inválidas: 0


,id,station_id,timestamp,temperature_c,humidity_pct,wind_speed_kmh,wind_direction_deg,pressure_hpa,precipitation_mm,visibility_km,uv_index
0,ccdcf0e5-6ca1-4d82-bc25-0881de96dc7c,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:00:00+00:00,32.4,68.9,23.1,45.7,1013.2,0.0,11.8,0.0
1,c1a1b04e-61c2-43f6-b478-70f9aa765c5b,4ca4c103-894d-4f59-a911-f3abb8077e3c,2025-09-01 00:00:00+00:00,31.9,68.8,22.2,41.6,1013.1,0.0,11.5,0.0
2,e89069bc-40ed-48c1-ad94-b0bb9c7dd556,5c5c4158-91cb-4073-ab4c-9bb10037dc16,2025-09-01 00:00:00+00:00,31.8,68.1,23.4,40.0,1013.0,0.0,11.3,0.0
3,509121ee-dc99-4a22-b77c-2a88a644753f,82a31d3a-9895-42e5-8ea0-11e8cb15b964,2025-09-01 00:00:00+00:00,31.3,68.9,21.3,47.3,1013.2,0.0,11.7,0.0
4,05b70544-1f18-4a3e-993a-b82b453bc12e,8f0185fa-a4c3-41c1-8f3b-d08f4f6aece0,2025-09-01 00:00:00+00:00,32.4,67.2,22.8,50.7,1013.1,0.0,12.0,0.0


## Eliminación de duplicados exactos

La definición indicada para el proyecto excluye solo `id`: dos filas con la misma estación, timestamp y medidas son la misma lectura aunque sus identificadores sean distintos. Ordenamos por estación y tiempo para que la primera observación conservada sea determinista.

El campo `id` identifica el registro almacenado, no la observación física. Por eso `duplicate_columns` contiene todas las columnas excepto `id`, y `duplicated(..., keep='first')` retiene una sola fila de cada grupo idéntico. Esta es exactamente la regla comunicada por el profesor.

In [4]:
df = df.dropna(subset=['station_id', 'timestamp']).sort_values(['station_id', 'timestamp', 'id'])
duplicate_columns = [column for column in df.columns if column != 'id']
exact_duplicate = df.duplicated(subset=duplicate_columns, keep='first')
duplicates_removed = int(exact_duplicate.sum())
duplicate_examples = df.loc[exact_duplicate | exact_duplicate.shift(fill_value=False)].head(4)
df = df.loc[~exact_duplicate].copy()

print(f'Duplicados exactos eliminados: {duplicates_removed:,}')
print(f'Filas tras deduplicar: {len(df):,}')
duplicate_examples

Duplicados exactos eliminados: 82,944
Filas tras deduplicar: 1,268,064


,id,station_id,timestamp,temperature_c,humidity_pct,wind_speed_kmh,wind_direction_deg,pressure_hpa,precipitation_mm,visibility_km,uv_index
808417,b35c59df-cf45-4718-ada6-1e29fb3ea278,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2026-04-30 22:00:00+00:00,31.0,71.5,21.2,45.2,1015.8,0.0,11.8,0.0
808441,36249dc2-f336-4e21-a849-e2db5d12163f,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2026-04-30 22:05:00+00:00,31.0,71.5,21.3,44.2,1015.8,0.0,11.9,0.0
808440,ac0a4e98-4a18-4ef3-b244-8f5af027d9b8,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2026-04-30 22:05:00+00:00,31.0,71.5,21.3,44.2,1015.8,0.0,11.9,0.0
808465,9549beb2-b7ef-449b-810f-e1136a066b67,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2026-04-30 22:10:00+00:00,31.0,71.2,21.7,44.3,1015.8,0.0,12.1,0.0


## Valores ausentes, rangos y anomalías

Los rangos imposibles se transforman en ausentes. Una ausencia se recupera por interpolación lineal solo cuando tiene una lectura válida inmediatamente antes y después, ambas a cinco minutos. No se interpolan huecos de red. Los saltos y ceros sospechosos se conservan con indicadores para no confundir un fenómeno real con un error de sensor.

### Valores inválidos

Humedad fuera de 0-100 %, dirección de viento fuera de 0-360 grados o precipitaciones negativas son ejemplos de valores imposibles. Se sustituyen por `NaN` para que el proceso pueda distinguirlos de un cero válido.

### Imputación limitada

Solo se imputa un valor si las lecturas anterior y posterior son de la misma estación, tienen valores disponibles y están exactamente a cinco minutos. La imputación es la media de ambas medidas. No se rellenan cortes de horas o días: la fila queda incompleta y se elimina del dataset limpio.

### Anomalías auditables

Los saltos rápidos se marcan con columnas `anomalous_jump_*`. Los ceros de temperatura, humedad, presión o visibilidad se señalan como sospechosos. No se eliminan porque podrían responder a una situación válida o requerir una decisión de dominio posterior. Los huecos de red se identifican con `gap_after_previous` e `irregular_interval`.

In [5]:
df = df.sort_values(['station_id', 'timestamp']).reset_index(drop=True)
invalid_range = pd.DataFrame(False, index=df.index, columns=NUMERIC_COLUMNS)
for column, (lower, upper) in VALID_RANGES.items():
    invalid_range[column] = ~df[column].between(lower, upper) & df[column].notna()
    df.loc[invalid_range[column], column] = np.nan

df['interval_since_previous'] = df.groupby('station_id')['timestamp'].diff()
df['gap_after_previous'] = df['interval_since_previous'].gt(EXPECTED_INTERVAL)
df['irregular_interval'] = (
    df['interval_since_previous'].notna()
    & df['interval_since_previous'].ne(EXPECTED_INTERVAL)
)

imputed_values = 0
imputed_cell = pd.DataFrame(False, index=df.index, columns=NUMERIC_COLUMNS)
for column in NUMERIC_COLUMNS:
    previous_value = df.groupby('station_id')[column].shift()
    next_value = df.groupby('station_id')[column].shift(-1)
    next_timestamp = df.groupby('station_id')['timestamp'].shift(-1)
    isolated_missing = (
        df[column].isna()
        & previous_value.notna()
        & next_value.notna()
        & df['interval_since_previous'].eq(EXPECTED_INTERVAL)
        & next_timestamp.sub(df['timestamp']).eq(EXPECTED_INTERVAL)
    )
    df.loc[isolated_missing, column] = (
        previous_value.loc[isolated_missing] + next_value.loc[isolated_missing]
    ) / 2
    imputed_cell.loc[isolated_missing, column] = True
    imputed_values += int(isolated_missing.sum())

for column, max_change in MAX_CHANGE_PER_INTERVAL.items():
    change = df.groupby('station_id')[column].diff().abs()
    df[f'anomalous_jump_{column}'] = (
        df['interval_since_previous'].eq(EXPECTED_INTERVAL) & change.gt(max_change)
    )

for column in ['temperature_c', 'humidity_pct', 'pressure_hpa', 'visibility_km']:
    df[f'suspicious_zero_{column}'] = df[column].eq(0)

anomaly_columns = [column for column in df if column.startswith(('anomalous_jump_', 'suspicious_zero_'))]
df['has_anomaly_flag'] = df[anomaly_columns].any(axis=1)
df['was_imputed'] = imputed_cell.any(axis=1)
unresolved_missing = df[NUMERIC_COLUMNS].isna().any(axis=1)
rows_removed_incomplete = int(unresolved_missing.sum())
clean_df = df.loc[~unresolved_missing].copy()

print(f'Mediciones fuera de rango: {int(invalid_range.to_numpy().sum()):,}')
print(f'Valores imputados: {imputed_values:,}')
print(f'Filas eliminadas por valores incompletos: {rows_removed_incomplete:,}')
print(f'Filas con una anomalía marcada: {clean_df["has_anomaly_flag"].sum():,}')

Mediciones fuera de rango: 0
Valores imputados: 0
Filas eliminadas por valores incompletos: 0
Filas con una anomalía marcada: 69


## Comparación antes y después

Mostramos el volumen de datos tras cada transformación. Las anomalías se conservan identificadas para su revisión en las etapas posteriores.

In [6]:
summary = pd.DataFrame({
    'etapa': ['Original', 'Tras deduplicar', 'Conjunto limpio'],
    'filas': [len(raw_df), len(df), len(clean_df)],
})
summary

,etapa,filas
0,Original,1351008
1,Tras deduplicar,1268064
2,Conjunto limpio,1268064


## Exportación y registro de transformaciones

Guardamos el conjunto limpio para las etapas posteriores y un informe JSON con el recuento de cada transformación. Se mantienen las columnas de trazabilidad y los indicadores de anomalía para que el modelado pueda excluirlas o analizarlas según el caso de uso.

Se generan tres artefactos:

- `aruba_readings_cleaned.csv`: tabla lista para análisis y modelado.
- `aruba_readings_cleaned.json`: la misma tabla en formato JSON.
- `data_cleaning_report.json`: registro de filas de entrada, duplicados eliminados, valores inválidos, imputaciones, huecos, anomalías y filas finales.

El resultado conserva `was_imputed`, `gap_after_previous`, `irregular_interval`, `has_anomaly_flag` y los indicadores específicos de salto o cero sospechoso. De este modo una etapa posterior puede filtrar anomalías sin perder la historia de las transformaciones aplicadas.

In [7]:
clean_df.to_csv(CSV_OUTPUT_PATH, index=False)
clean_df.to_json(JSON_OUTPUT_PATH, orient='records', date_format='iso')

report = {
    'input_rows': int(len(raw_df)),
    'rows_without_valid_key': int(len(raw_df) - len(df) - duplicates_removed),
    'exact_duplicates_removed': duplicates_removed,
    'invalid_measurements': int(invalid_range.to_numpy().sum()),
    'isolated_values_imputed': imputed_values,
    'rows_removed_with_unresolved_missing_values': rows_removed_incomplete,
    'network_gaps_detected': int(df['gap_after_previous'].sum()),
    'rows_with_anomaly_flag': int(clean_df['has_anomaly_flag'].sum()),
    'output_rows': int(len(clean_df)),
}
with open(REPORT_OUTPUT_PATH, 'w', encoding='utf-8') as report_file:
    json.dump(report, report_file, indent=2)

print(json.dumps(report, indent=2))
print(f'CSV limpio: {CSV_OUTPUT_PATH}')
print(f'JSON limpio: {JSON_OUTPUT_PATH}')
print(f'Registro de limpieza: {REPORT_OUTPUT_PATH}')

{
  "input_rows": 1351008,
  "rows_without_valid_key": 0,
  "exact_duplicates_removed": 82944,
  "invalid_measurements": 0,
  "isolated_values_imputed": 0,
  "rows_removed_with_unresolved_missing_values": 0,
  "network_gaps_detected": 36,
  "rows_with_anomaly_flag": 69,
  "output_rows": 1268064
}
CSV limpio: ../data/processed/0_cleaning/aruba_readings_cleaned.csv
JSON limpio: ../data/processed/0_cleaning/aruba_readings_cleaned.json
Registro de limpieza: ../data/processed/0_cleaning/data_cleaning_report.json
